## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [ ]:
import seaborn as sns
import pandas as pd


titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
625,0,1,male,61.0,0,0,32.3208,S,First,man,True,D,Southampton,no,True
203,0,3,male,45.5,0,0,7.2250,C,Third,man,True,NaN,Cherbourg,no,True
734,0,2,male,23.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
424,0,3,male,18.0,1,1,20.2125,S,Third,man,True,NaN,Southampton,no,False
378,0,3,male,20.0,0,0,4.0125,C,Third,man,True,NaN,Cherbourg,no,True


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [50]:
missed_data_count = titanic_data.isnull().sum()
print(missed_data_count)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [51]:
delete_columns_data = titanic_data.loc[:, titanic_data.isnull().sum() <= len(titanic_data) / 2]

titanic_data = delete_columns_data.dropna(thresh=len(delete_columns_data.columns) // 2 + 1, axis="rows")

print(titanic_data.isnull().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
embark_town      2
alive            0
alone            0
dtype: int64


### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [52]:
median_man = titanic_data[titanic_data['who'] == 'man']['age'].median().round()
median_woman = titanic_data[titanic_data['who'] == 'woman']['age'].median().round()
median_child = titanic_data[titanic_data['who'] == 'child']['age'].median().round()

titanic_data.loc[(titanic_data['age'].isna()) & (titanic_data['who'] == 'man'), 'age'] = median_man
titanic_data.loc[(titanic_data['age'].isna()) & (titanic_data['who'] == 'woman'), 'age'] = median_woman
titanic_data.loc[(titanic_data['age'].isna()) & (titanic_data['who'] == 'child'), 'age'] = median_child

print(titanic_data.isnull().sum())

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       2
class          0
who            0
adult_male     0
embark_town    2
alive          0
alone          0
dtype: int64


### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [64]:
titanic_data = titanic_data.dropna(thresh=len(titanic_data.columns) - 1)
titanic_data[titanic_data.isnull().any(axis="columns")]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone


### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [ ]:
city_counts = titanic_data['embark_town'].value_counts()

most_common_city = titanic_data['embark_town'].value_counts().idxmax()

print(f"Город, из которого отправилось больше всего пассажиров: {most_common_city}")

Город, из которого отправилось больше всего пассажиров: Southampton


### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [70]:
survivors_count = titanic_data['survived'].sum()

survivors_percent = round((survivors_count / len(titanic_data)) * 100, 2)

print(f"Процент выживших: {survivors_percent}%")

Процент выживших: 38.25%


### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [84]:
survivors = titanic_data[titanic_data["survived"] == 1]
survivors_by_town = survivors["embark_town"].value_counts()
survivors_by_town = survivors_by_town.reindex(titanic_data["embark_town"].unique(), fill_value=0) # дополнение нулями

print(survivors_by_town)

embark_town
Southampton    217
Cherbourg       93
Queenstown      30
Name: count, dtype: int64


### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [ ]:
survival_by_class = titanic_data.groupby('class')['survived'].mean() * 100
survival_by_class = round(survival_by_class, 2)

print(survival_by_class)

class
First     62.62
Second    47.28
Third     24.24
Name: survived, dtype: float64


### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [95]:
rich_passengers = titanic_data[titanic_data['fare'] >= 100]

rich_count = len(rich_passengers)

rich_survivors = rich_passengers['survived'].sum()

rich_survivors_percent = round((rich_survivors / rich_count) * 100, 2)

print(rich_survivors_percent)

73.58


### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [98]:
alone_children = titanic_data[(titanic_data['who'] == 'child') & (titanic_data['alone'])]

count_children_alone = len(alone_children)

print(count_children_alone)

6


Какие выводы вы можете сделать о выживших пассажирах Титаника? 